In [6]:
%load_ext autoreload 
%autoreload 2

In [43]:
from pomap_v2 import pomap
import polars as pl
import datetime as dt

import numpy as np
from functools import reduce


### Before we get started, let's prototype the functional behaviour we want on a simpler problem

In [41]:
from typing import Self

class Operation:
    def __init__(self, nodes: list[Self]):
        self.nodes = nodes

    def operate(self, x):
        current = x
        for node in self.nodes:
            current = node.operate(current)
        return current

    def compose(self, other: Self):
        return Operation(self.nodes + other.nodes)

    def __sum__(self, other: Self):
        return self.compose(other)

class PrimitiveOperation(Operation):

    def __init__(self):
        self.nodes = [self]


class Add2(PrimitiveOperation):

    def operate(self, x):
        return x + 2


In [42]:
add2 = Add2()
add4 = add2.compose(add2)

In [40]:
add4.operate(1)

5

### Let's start prototyping a new PoMap

In [185]:
# First, a sketch, a Pomap is defined by a 'dimension' and a set of expected labels     

class _Pomap:

    # A PoMap is defined by a 'dimension' and a set of labels belonging to that dimension
    def __init__(self, nodes: list[Self], name: str):

        self.name = name
        self._nodes = nodes

        # Implement some standardised naming for the subclasses to use
        self._train_column_name = lambda label:  f'train({self.name}={label})'
        self._test_column_name = lambda label: f'test({self.name}={label})'
        self._validate_column_name = lambda label: f'validate({self.name}={label})'

    def __repr__(self):
        return self.name


    def __getitem__(self, arg: str):
        for node in self._nodes:
            if node.name == 'arg':
                return node
        raise ValueError(f'Pomap has no node {arg}')
                
# -=-=-=-=-=-=-=--=-=-==-=-=-=-=-=--=-=-=-=-=-=-=-=-=-=-=-=-==-=-=-=-=-=-=-=-=-=--=-
#   After this, things get interesting, since we start to deal with how the pomap actually behaves
# COMPOSITION


    def product(self, other: Self):
        # This composition assumes that ONLY the product operation is possible, not the sum.

        overlapping_names = set(self._nodes).intersection(other._nodes) 
        assert overlapping_names == set(), f"Cannot compose two Pomaps with overlapping names. Found {overlapping_names} in common"
        return _Pomap(nodes=self._nodes + other._nodes, name=f'{self.name} + {other.name}')

    @property
    def labels(self) -> pl.DataFrame:
        # TODO this will have to change if we introduce a product operation
        # (or any other composition)
        # Since, we will need to recurse through the syntax tree and 
        # pluck out the labels of all the child nodes

        leaf_nodes = self._find_leaf_nodes(self)               
        leaf_labels = [node.labels for node in leaf_nodes]

        df = reduce(lambda left, right: left.to_frame().join(right.to_frame(), how='cross'),  leaf_labels)

        return df


    @staticmethod
    def _find_leaf_nodes(node: _Pomap):
        if (len(node._nodes) == 1) and (node._nodes[0] is node):
            return [node]
        
        leaf_nodes = []
        for child in node._nodes:
            leaf_nodes.extend(_Pomap._find_leaf_nodes(child))
        
        return leaf_nodes

    # Need to implement these in terms of the composed logic
    def label_rows_as_train(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        pass

    def label_rows_as_test(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        pass

    def label_rows_as_validate(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        pass 

#### Model Interface
    def label_to_train(self, df: pl.DataFrame, label: dict):
        df = self.label_rows_as_train(df, label)
        df = df.filter(
            self.train_column_name(label)
        )
        df = df.drop(
            self.train_column_name(label)
        )

        return df

In [186]:
class Pomap(_Pomap):

    def __init__(self, name: str):
        super().__init__(nodes=[self], name=name)

    @property
    def labels(self, other: Self) -> pl.Series:
        raise NotImplementedError

    # These three (train, test, validate) functions define the behaviour of the PoMap.
    # E.g, is it a cross validation, is it categorical, etc.
    # See below for an example of a reasonably complex example.
    def label_rows_as_train(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        raise NotImplementedError

    # There has to be a separate one for test and validation, because
    # train and test data must be distinct.
    def label_rows_as_test(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        raise NotImplementedError

    # .... as above
    def label_rows_as_validate(self, df: pl.DataFrame, label: dict) -> pl.DataFrame:
        raise NotImplementedError

In [187]:
class CategoricalPomap(Pomap):

    def __init__(self, column: str, labels: list):
        super().__init__(name=f"Categorical {column}: {labels}")
        self._column = column
        self._labels = labels

    @property
    def labels(self) -> pl.Series:
        return pl.Series(values=self._labels, name=self.name)

    def _label_rows_as(self, df, label, column_name) -> pl.DataFrame:

        df =  df.with_columns(
            (pl.col(self.column) == label).alias(column_name)
        )

        return df
    
    def label_rows_as_train(self, df: pl.DataFrame, label) -> pl.DataFrame:
        return self._label_rows_as(df, label, self._train_column_name(label))

    def label_rows_as_test(self, df: pl.DataFrame, label) -> pl.DataFrame:
        return self._label_rows_as(df, label, self._test_column_name(label))

    def label_rows_as_validate(self, df: pl.DataFrame, label) -> pl.DataFrame:
        return self._label_rows_as(df, label, self._validate_column_name(label))
        
        

#### Example time

In [188]:
example_df = pl.from_dict(
    {'cat1': ['a',  'b',],
    'cat2': ['c', 'd', ]}
)

timestamps = pl.datetime_range(dt.datetime(2025, 1, 1),
                               dt.datetime(2025, 1, 7),
                               interval='1h',
                               eager=True
                               ).rename('ts').to_frame()

example_df = example_df.join(timestamps, how='cross')
example_df = example_df.with_columns(
    feature = np.random.normal(0, 1, example_df.shape[0])
)

example_df = example_df.with_columns(hour=pl.col('ts').dt.hour())

In [189]:
pmap_cat1 = CategoricalPomap(column='cat1', labels=['a', 'b'])

In [190]:
pmap_cat2 = CategoricalPomap(column='cat2', labels=['c', 'd'])

In [191]:
composed = pmap_cat1.product(pmap_cat2)

In [192]:
pmap_cat1.labels

"Categorical cat1: ['a', 'b']"
str
"""a"""
"""b"""


In [193]:
composed.labels

"Categorical cat1: ['a', 'b']","Categorical cat2: ['c', 'd']"
str,str
"""a""","""c"""
"""a""","""d"""
"""b""","""c"""
"""b""","""d"""


In [55]:
# An example of a child class.

class RandomisedCrossValidationPoMap(Pomap):

    def __init__(self, dims, folds: int, index_column: 'str'):
        super().__init__(name=f'RandomisedCrossValidation:{index}')
        self.fold_labels = [str(c) for c in range(folds)]
    
    def label_rows_as_test(self, df: pl.DataFrame, label: dict):
        assert df.unique(self.index_column).shape[0] == df.shape[0]
        index_to_test_fold = {}
        for index in df.select(self.index_column).unique():
            index_to_test_fold[index] = np.random.choice(self.fold_labels)

        df = df.with_columns(
            pl.col(self.index_column)
            .replace_strict(index_to_test_fold)
            .alias(self.train_column_name(label, self.dims))
        )

        return df

    def label_rows_as_train(self, df, label):
        df = self.label_rows_as_test(df, label)
        test_column = self.test_column_name(label, self.dims)

        # A node is in test if it's label did NOT appear in the train set.
        df = df.with_columns(
            ~(pl.col(test_column))
            .alias(self.train_column_name(label, self.dims))
        )
        df = df.drop(test_column)

        return df

    @property
    def labels(self) -> pl.Series:
        return pl.Series(self.fold_labels, name=name)
        

5